> 在线讲义：[第 20 章 · 潜空间世界模型：VLA-JEPA，看视频学"世界怎么变"，但别偷看答案](https://wikieden.github.io/vla-from-scratch/lessons/20-latent-world-model.html) · 本书《VLA 从零到 π0》

# 第 20 章 · 潜空间世界模型：VLA-JEPA，看视频学"世界怎么变"，但别偷看答案

配套讲义：仓库里 `vla-deep-dive/notes/lessons/20-latent-world-model.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 2 步 | `expected_pixel_loss(move, n_bg, sigma)`、`latent_loss(move)` | 像素 26（背景 25，占 96%）；潜空间 1 |
| 第 2 步 | `simulate_losses(...)` | 随机抽样后接近 26 和 1 |
| 第 3 步 | `leaky_reconstruction(...)` | 看得到未来时误差 0，和动作无关 |
| 第 4 步 | `JEPA_TABLES` | 97.2 vs 96.1；79.5 vs 62.9 |

玩具设定是编的：一帧 = 1 个物体位置 + n_bg 个背景像素；表 1、表 3 的数取自 vla-jepa-2602.10098.txt。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
JEPA_TABLES = {"LIBERO 平均": (97.2, 96.1), "LIBERO-Plus 平均": (79.5, 62.9)}  # (有人手视频, 没有)

**函数 `expected_pixel_loss`**：像素损失的期望：物体部分 move²，背景部分 n_bg·σ²（模型猜不到相机抖动）。

In [ ]:
def expected_pixel_loss(move, n_bg, sigma):
    """像素损失的期望：物体部分 move²，背景部分 n_bg·σ²（模型猜不到相机抖动）。"""
    return move**2 + n_bg * sigma**2, move**2, n_bg * sigma**2

**函数 `latent_loss`**：理想编码器只留物体位置：潜空间里只剩物体那一项。

In [ ]:
def latent_loss(move):
    """理想编码器只留物体位置：潜空间里只剩物体那一项。"""
    return move**2

**函数 `simulate_losses`**：抽样验证：预测"下一帧 = 这一帧"，真实下一帧物体移动 move、背景随机抖动。

In [ ]:
def simulate_losses(move, n_bg, sigma, trials=20000, seed=0):
    """抽样验证：预测"下一帧 = 这一帧"，真实下一帧物体移动 move、背景随机抖动。"""
    rng = np.random.default_rng(seed)
    obj_err = np.full(trials, float(move))  # 最朴素的预测"下一帧和这一帧一样"：物体误差就是移动量
    bg_err = rng.normal(0, sigma, size=(trials, n_bg))
    pixel = obj_err**2 + (bg_err**2).sum(axis=1)
    return float(pixel.mean()), float((obj_err**2).mean())

**函数 `leaky_reconstruction`**：隐变量直接等于下一帧（看得到未来）：不管动作是什么，重建误差都是 0。

In [ ]:
def leaky_reconstruction(frames_next, actions):
    """隐变量直接等于下一帧（看得到未来）：不管动作是什么，重建误差都是 0。"""
    z = np.asarray(frames_next, float)
    recon = z.copy()
    del actions  # 动作根本没用上
    return float(((recon - frames_next) ** 2).mean())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 像素损失 vs 潜空间损失（物体移动 1、背景 100 像素、抖动 σ = 0.5）")
total, obj, bg = expected_pixel_loss(1, 100, 0.5)
print(f"  像素损失期望 {total:.0f}（物体 {obj:.0f} + 背景 {bg:.0f}，背景占 {bg / total:.0%}）")
print(f"  潜空间损失 {latent_loss(1):.0f}")
sim_pixel, sim_latent = simulate_losses(1, 100, 0.5)
print(f"  抽样：像素 {sim_pixel:.2f}，潜空间 {sim_latent:.2f}")

In [ ]:
print("\n第三步 信息泄露：隐变量抄下一帧")
rng = np.random.default_rng(1)
nxt = rng.normal(size=(4, 101))
for acts in (np.zeros(4), rng.normal(size=4)):
    print(f"  动作 {np.round(acts, 2).tolist()}：重建误差 {leaky_reconstruction(nxt, acts):.1f}")

In [ ]:
print("\n第四步 人手视频的作用")
for k, (w, wo) in JEPA_TABLES.items():
    print(f"  {k}：有 {w}% / 无 {wo}%，差 {w - wo:.1f}")

## 3. 代码里才看得到的

1. `simulate_losses` 里的预测用最朴素的"下一帧和这一帧一样"：物体那项误差是移动量的平方，背景那项全是猜不到的相机抖动。像素损失里背景占大头，潜空间里背景被编码器丢掉了。
2. `leaky_reconstruction` 让隐变量直接等于下一帧；不管喂进去的动作是什么，重建都完美，这就是"答案纸"。
3. 编码器在代码里就是"只取物体那一维"，是最理想的 F；真实的 V-JEPA2 不会把背景压到正好 0。

## 4. 自己改一改

- 把背景抖动改成"整体平移"（所有像素同加一个数），看像素损失怎么变。
- 给 `latent_loss` 的编码器混入 5% 的背景信息，看潜空间损失里背景占多少。
- 用 `two_prop_z`（第 19 章）检验 97.2% vs 96.1%、79.5% vs 62.9%。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看视频学做菜，别学镜头怎么晃；
> 先翻译成"菜什么样"再比，像素抖动就不算数；
> 答案纸只给老师看，学生只看现在；
> 推理时陪练下场；人手视频不教新招，教你换个地方也稳。